# KG catch-up + intent chat session -> turns + knowledge graph

A new process flow on top of **[`kg_intent_chat.ipynb`](kg_intent_chat.ipynb)**: instead of
starting cold, the agent first figures out **how long it's been since it last spoke with this
human, what they'd talked about back then, and how MUCH of that it should expect to hear about
again** -- then keeps the conversation going, topic by topic, until it judges it has *enough* for
the catch-up period, not just "asked once about everything."

Built on **[`gaps_from_kg/get_temporal_containers.py`](../src/cltl/gaps_from_kg/get_temporal_containers.py)**
-- a different query layer over the same GraphDB repository (via `cltl.brain.LongTermMemory`)
than `kg_gap_finder.py`/`intent_gap_finder.py`'s rdflib/SPARQL-endpoint queries, which is what the
rest of the per-turn flow below still uses unchanged. See
**[`catch_up_from_kg.py`](catch_up_from_kg.py)** for the full implementation; in short, a loop:

1. **Ask about a catch-up-period topic** (`SaturationTracker.next_question()`) -- one of
   `chat_sessions.DEFAULT_GAP_ACTIVITY_TYPES` (exercise, diet, sleep, symptoms, medication, ...)
   this human has real history with, but not yet ENOUGH reported for this period (see
   "Saturation" below).
2. **Whatever they report is handled entirely by `KgIntentChatSession`'s own EXISTING per-turn
   flow, unchanged**: SRL extraction -> push to the KG -> `intent_gap_finder.next_intent_gap()`
   keeps asking its own follow-up questions about *that* activity (what/how much/when/where) for
   as long as its matching intent still has unmet requirements.
3. **Once that activity's own follow-ups are exhausted**, go back to step 1 -- another
   under-covered topic, or the same one again if it's still short -- **unless every topic has
   reached saturation**, at which point (see "Wrapping up" below) the LLM decides how to close
   things out, and the conversation continues normally after that.

**The catch-up period is capped at `catch_up.MAX_SATURATION_GAP_DAYS` (14 days)**, even when the
real gap since the last conversation is much longer -- a human who hasn't talked in two months
still only needs the last two weeks caught up on live, not the whole two months.

**Saturation** -- "enough knowledge for the catch-up period" -- is defined *per topic* against a
fixed **weekly** baseline (`catch_up.CATCH_UP_WINDOW_DAYS`, 7 days), not one tied to however long
the catch-up period itself is: `_windowed_average_rate()` tiles that week-long window backwards
across this human's *entire* history to get their typical weekly rate, which is then scaled to
however many days the (capped) catch-up period covers. If they've historically reported exercise
~3 times a week and the catch-up period is 2 weeks, 6 reported live is enough for exercise
specifically -- not "ask about it once and move on" and not "keep asking forever." A hard
per-topic cap (`SaturationTracker.MAX_ASKS_PER_TOPIC`, default 3) still gives up on a topic the
human simply has nothing more to say about, regardless of its target.

**Wrapping up.** The FIRST time every topic is saturated (or capped out),
`SaturationTracker.wrap_up_message()` sends one message naming what was actually covered and
lets the LLM decide for itself: continue with a short question if there's an obvious thread left,
or wrap up warmly and say goodbye. After that one message, replies fall through to the plain
default `agent_fn` unchanged, so the conversation continues (or ends, if the human says goodbye
back) like an ordinary chat.

**Before running this:** same requirements as `kg_intent_chat.ipynb` -- `OPENAI_API_KEY` set, and
a reachable knowledge-graph SPARQL endpoint (e.g. a local GraphDB `event_sandbox` repository) at
`KG_ADDRESS` below, ideally one that already has some history for `HUMAN` in it (otherwise there's
nothing to catch up on, and this degrades to a plain, short "what's new?" opener with an
immediately-saturated tracker).

In [ ]:
import time
from datetime import datetime, timedelta

from chat_sessions import KgIntentChatSession, openai_agent, save_turns
import catch_up_from_kg as catch_up

KG_ADDRESS = "http://localhost:7200/repositories/event_sandbox"
KG_LOG_DIR = "kg_logs"
HUMAN = "Mehmet"
HUMAN = "Jan"

# Directory of intent *.json files -- None auto-discovers the project's own intents/ folder (see
# intent_gap_finder.load_intents()/_default_intents_dir()).
INTENTS_DIR = None

# A FRESH id every run, not a fixed constant -- see kg_intent_chat.ipynb's own CHAT_ID cell for
# why (reusing one across runs collides every run's activities on the same subject URIs).
CHAT_ID = int(time.time())

# "Now", for both the catch-up queries below and every activity pushed to the graph this run.
CURRENT_DATE = datetime.now()

# Used only if the KG has no earlier conversation with HUMAN on record at all (e.g. a brand new
# human, or a fresh/empty graph) -- how far back to assume the last (nonexistent) conversation
# was, so there's still a sensible "it's been N days" opener instead of a crash.
FALLBACK_LAST_CONVERSATION_DATE = CURRENT_DATE - timedelta(days=7)


## Step 1: how long has it been, and what's worth catching up on?

Connects to the same KG `KG_ADDRESS` points at (via `cltl.brain.LongTermMemory` this time, not
rdflib). `connect_brain()` also makes sure the graph has
**[`gaps_from_kg/n2mu_sem_roles.py`](../src/cltl/gaps_from_kg/n2mu_sem_roles.py)**'s small
`rdfs:subPropertyOf` mapping uploaded (a one-time, idempotent step -- see that module's own
docstring): without it, every real activity looks dateless/actorless/placeless to
`get_temporal_containers()`'s underlying query, since activities only ever carry this project's
own `n2mu:agent`/`n2mu:location`/`n2mu:time/...` predicates, never `sem:hasActor`/`hasPlace`/
`hasTime` directly. Past that one-time schema write, the rest of this step only reads.

`find_catch_up_topics()` caps the period it actually tries to catch up on at
`MAX_SATURATION_GAP_DAYS` days (see the intro above), and calibrates each topic's target against
a fixed *weekly* baseline (`weekly_rate`) computed from this human's full history before that
period -- not against the (possibly much longer) real gap itself.

In [2]:
brain = catch_up.connect_brain(KG_ADDRESS, log_dir=KG_LOG_DIR)

last_conversation_date = catch_up.find_last_conversation_date(
    HUMAN, brain, CURRENT_DATE, FALLBACK_LAST_CONVERSATION_DATE
)
catch_up_topics = catch_up.find_catch_up_topics(brain, CURRENT_DATE, last_conversation_date)

gap_days = (CURRENT_DATE.date() - last_conversation_date.date()).days
capped_days = min(gap_days, catch_up.MAX_SATURATION_GAP_DAYS)
print(f"Last conversation with {HUMAN}: {last_conversation_date} ({gap_days} day(s) ago)")
print(f"Catching up on the last {capped_days} day(s)"
      + (f" (capped from the full {gap_days}-day gap)" if capped_days < gap_days else ""))
print(f"Found {len(catch_up_topics)} catch-up topic(s):")
for topic in catch_up_topics:
    print(f"  - {topic['activity_type']}: {topic['weekly_rate']:.1f}/week historically -> "
          f"target {topic['expected_count']} for this period (already "
          f"{topic['initial_reported_count']} logged), most recently "
          f"\"{topic['latest_label']}\" on {topic['latest_date']}")


2026-09-18 08:56:50 -     INFO -                                    cltl.brain.LongTermMemory - Booted
2026-09-18 08:56:50 -     INFO -                                  cltl.brain.ThoughtGenerator - Booted
2026-09-18 08:56:50 -     INFO -                                  cltl.brain.LocationReasoner - Booted
2026-09-18 08:56:50 -     INFO -                                      cltl.brain.TypeReasoner - Booted
2026-09-18 08:56:50 -     INFO -                                   cltl.brain.TrustCalculator - Booted


Our previous conservation was on Friday 2026-09-11 08:56:45.855626
Today is Friday 2026-09-18 08:56:45.855626
What happened in the last 7 days?
PREFIX n2mu: <http://cltl.nl/leolani/n2mu/>            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>            select ?id ?label where {                ?id rdf:type n2mu:exercise.                 ?id rdfs:label ?label .                 }
I found 29 activities
PREFIX n2mu: <http://cltl.nl/leolani/n2mu/>            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>            select ?id ?label where {                ?id rdf:type n2mu:take_food.                 ?id rdfs:label ?label .                 }
I found 13 activities
PREFIX n2mu: <http://cltl.nl/leolani/n2mu/>            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>            select ?id ?label where {                ?id rdf:type n2mu:take_drink.                 ?id rdfs:label ?label .                 }
I found 17 activities
PREFIX n2mu: <http://cltl.nl/leola

## Step 2: open the chat

`SaturationTracker.opening_question()` turns the findings above into the agent's very first turn
(recorded via `open_with()`, same as `kg_chat_session.ipynb`'s own opening-greeting cell -- see
there for why it's a real, annotated-and-pushed turn rather than just printed text), naming the
`LEAD_TOPICS` topics with the biggest shortfall as concrete memory prompts, and marks those as
asked once so the loop below doesn't immediately repeat them.

`wrap_agent_fn_with_saturation_loop()` then wraps the plain `openai_agent()` default reply
function: for as long as `tracker.is_saturated()` is False, a reply that would otherwise be the
generic default LLM one asks about the next under-covered topic instead -- see
`catch_up_from_kg.py`'s own module docstring for exactly when that fires versus an ordinary
intent-driven gap question, and how `on_new_subject=tracker.record_new_activity` keeps the
tracker's own counts up to date as the human reports things live.

In [3]:
from kg_chat_gui import run_gui

LEAD_TOPICS = 2
tracker = catch_up.SaturationTracker(catch_up_topics, human=HUMAN)
agent_fn = catch_up.wrap_agent_fn_with_saturation_loop(openai_agent(), tracker)

kg_session = KgIntentChatSession(
    chat=CHAT_ID,
    human=HUMAN,
    kg_address=KG_ADDRESS,
    log_dir=KG_LOG_DIR,
    intents_dir=INTENTS_DIR,
    agent_fn=agent_fn,
    on_new_subject=tracker.record_new_activity,
)
print(f"Loaded {len(kg_session.intents)} intent(s) covering activity types: "
      f"{sorted({t for intent in kg_session.intents for t in intent.get('activity_types', [])})}")

opening_question = tracker.opening_question(CURRENT_DATE, last_conversation_date, lead_topics=LEAD_TOPICS)
print("saturation targets:", {t: (tracker.reported[t], target["expected_count"]) for t, target in tracker.targets.items()})

kg_session.open_with(opening_question)
kg_turns = run_gui(kg_session)


Loaded 15 intent(s) covering activity types: ['economic_condition', 'exercise', 'measurement', 'mental_condition', 'physical_condition', 'sleep', 'social_condition', 'symptom', 'take_drink', 'take_food', 'take_medicine']


2026-09-18 08:57:00 -     INFO -                                                       httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


saturation targets: {'mental_condition': (0, 1), 'diet': (2, 1), 'exercise': (10, 1), 'physical_condition': (10, 1), 'measurement': (16, 1), 'take_food': (12, 1), 'symptom': (3, 1)}
turn {'chat': 1789714605, 'human': 'Mehmet', 'date': '2026,Sep,18', 'turn': 1, 'speaker': 'agent', 'utterance': 'Hey Mehmet, it’s nice to talk with you again—it’s been a little while since our last chat. How have things been over the past week, especially with your mood, stress levels, and overall mental state?'}


2026-09-18 08:57:08 -     INFO -                                                       httpx2 - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-18 08:57:08 -     INFO -                                    cltl.brain.LongTermMemory - Booted
2026-09-18 08:57:08 -     INFO -                                  cltl.brain.ThoughtGenerator - Booted
2026-09-18 08:57:08 -     INFO -                                  cltl.brain.LocationReasoner - Booted
2026-09-18 08:57:08 -     INFO -                                      cltl.brain.TypeReasoner - Booted
2026-09-18 08:57:08 -     INFO -                                   cltl.brain.TrustCalculator - Booted


Compliance issues for chat 1789714605 turn 1:
 - extraction[0]: activity offset auto-corrected for 'been over the past week': (108,22) -> (107,23)
 - extraction[0]: experiencer offset auto-corrected for 'things': (103,6) -> (100,6)
 - extraction[0]: time offset auto-corrected for 'the past week': (112,13) -> (117,13)
 - extraction[1]: activity offset auto-corrected for 'mood': (143,4) -> (153,4)
 - extraction[1]: experiencer offset auto-corrected for 'your': (138,4) -> (148,4)
 - extraction[2]: activity offset auto-corrected for 'stress levels': (149,13) -> (159,13)
 - extraction[2]: experiencer offset auto-corrected for 'your': (138,4) -> (148,4)
 - extraction[3]: activity offset auto-corrected for 'overall mental state': (168,20) -> (178,20)
 - extraction[3]: experiencer offset auto-corrected for 'your': (138,4) -> (148,4)
Total nr of scenarios 1


...

Conversation id 1789714605 Total number of capsules extracted for this conversation 1


2026-09-18 08:58:36 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: talking with you next time_agent_agent [activity or social or nl/eckg/EventSeries_->_person])
2026-09-18 08:58:36 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: talking with you next time_patient_Mehmet [activity or social or nl/eckg/EventSeries_->_person])
2026-09-18 08:58:36 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: talking with you next time_participant_Mehmet [activity or social or nl/eckg/EventSeries_->_person])
2026-09-18 08:58:36 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: talking with you next time_time_today [activity or social or nl/eckg/EventSeries_->_point])
2026-09-18 08:58:36 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: talking with you next time_time

chat 1789714605 out of  1 turn 7 out of 1 turns


100%|██████████| 1/1 [00:00<00:00,  1.87it/s]


[turn 7] agent: I’m glad we could check in on all those areas together, Mehmet. Unless there’s anything else on your mind today, I’ll let you go for now and I’m looking forward to talking with you next time—take care.
    pushed 6 triple(s):
      talking with you next time  agent  =  I
      talking with you next time  patient  =  you
      talking with you next time  participant  =  Mehmet
      talking with you next time  time  =  today
      talking with you next time  time  =  for now
      talking with you next time  time  =  next time
[kg_chat_gui] conversation saved to /Users/piek/Desktop/Leolani/cltl-kg-driven-chat/chat_logs/chat1789714605_turns_20260918-090029.json
[kg_chat_gui] statistics saved to  /Users/piek/Desktop/Leolani/cltl-kg-driven-chat/chat_logs/chat1789714605_stats_20260918-090029.json
[kg_chat_gui] gap log saved to     /Users/piek/Desktop/Leolani/cltl-kg-driven-chat/chat_logs/chat1789714605_gaplog_20260918-090029.json


Inspect what was extracted, pushed, and where each agent reply came from:

In [4]:
print(f"{len(kg_session.turns)} turns, {len(kg_session.annotations)} annotated, "
      f"{len(kg_session.kg_pushes)} pushes to the knowledge graph")
print("reply sources:", kg_session.reply_sources)
print("saturated:", tracker.is_saturated(), "| wrapped up:", tracker.wrapped_up)
print("reported vs. target, per topic:",
      {t: (tracker.reported[t], target["expected_count"]) for t, target in tracker.targets.items()})
print("catch-up questions actually asked:", tracker.asked_log)
kg_session.kg_pushes


11 turns, 10 annotated, 10 pushes to the knowledge graph
reply sources: ['gap', 'gap', 'gap', 'gap', 'gap']
catch-up topics actually asked about: []


[{'conversations': 1, 'capsules': 2},
 {'conversations': 1, 'capsules': 2},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1},
 {'conversations': 1, 'capsules': 1}]

`kg_session.turn_log` has the same per-turn breakdown `kg_intent_chat.ipynb` prints live: what
each turn pushed, which intent (if any) matched, and which requirement its reply was about
(`None` for a default- or saturation-loop-driven reply -- both are indistinguishable from
`turn_log`'s own point of view, since neither comes from `intent_gap_finder`;
`kg_session.reply_sources`/`tracker.asked_log` above are what tell them apart).

In [5]:
kg_session.turn_log

[{'turn': 1,
  'speaker': 'agent',
  'utterance': 'Hi Mehmet, it’s nice to talk with you again—it’s been about a week since we last checked in. How have things been going with your eating lately, and how have your blood sugar measurements been over the past few days?',
  'triples_pushed': [{'subject': 'eating',
    'predicate': 'agent',
    'object': 'we'},
   {'subject': 'eating', 'predicate': 'participant', 'object': 'Mehmet'},
   {'subject': 'eating', 'predicate': 'qualification', 'object': 'lately'},
   {'subject': 'blood sugar measurements',
    'predicate': 'agent',
    'object': 'we'},
   {'subject': 'blood sugar measurements',
    'predicate': 'participant',
    'object': 'Mehmet'},
   {'subject': 'blood sugar measurements',
    'predicate': 'time',
    'object': 'over the past few days'}],
  'gap_queries': [],
  'selected_gap': None},
 {'turn': 2,
  'speaker': 'Mehmet',
  'utterance': 'I went out with a friend and had a few drinks',
  'triples_pushed': [{'subject': 'went out w

Save the turns, plus an **intent log** under `notebooks/intents_log/` summarizing the whole
session: the gap and the topics identified before the chat began (`topics_at_start`), how each of
those topics actually fared live (`topics_covered` -- reported/asked counts, whether its target
was met or it was capped out, and the tracker's own `asked_log`), and every
`intent_gap_finder.py` intent the per-turn flow consulted along the way (`intents_covered`) --
see `catch_up_from_kg.save_intent_log()` for the exact shape.

In [6]:
save_turns(kg_session.turns, "kg_catchup_intent_turns.json")
catch_up.save_intent_log(kg_session, tracker, catch_up_topics, CURRENT_DATE, last_conversation_date)


Wrote 11 turns to kg_catchup_intent_turns.json
